In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
df = pd.read_csv("data/historical_automobile_sales.csv" if Path("data/historical_automobile_sales.csv").exists() else "../data/historical_automobile_sales.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["Month_Num"] = df["Date"].dt.month
df["Month_Sin"] = np.sin(2*np.pi*df["Month_Num"]/12)
df["Month_Cos"] = np.cos(2*np.pi*df["Month_Num"]/12)

In [ ]:
features = ["Consumer_Confidence","Seasonality_Weight","Price","Advertising_Expenditure",
 "Competition","GDP","Growth_Rate","unemployment_rate","Month_Sin","Month_Cos",
 "Vehicle_Type","City"]
target = "Recession"
X, y = df[features], df[target]
categorical = ["Vehicle_Type","City"]
numeric = [c for c in features if c not in categorical]
pre = ColumnTransformer([
 ("num", Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]), numeric),
 ("cat", Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                   ("onehot",OneHotEncoder(handle_unknown="ignore"))]), categorical)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=.25, random_state=42, stratify=y)
models = {
 "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
 "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                           class_weight="balanced", random_state=42)
}
for name, model in models.items():
 pipe = Pipeline([("preprocess",pre),("model",model)])
 pipe.fit(X_train,y_train)
 pred = pipe.predict(X_test)
 prob = pipe.predict_proba(X_test)[:,1]
 print("\n", name)
 print(classification_report(y_test,pred))
 print("ROC-AUC:", roc_auc_score(y_test,prob))
 print("Confusion matrix:\n",confusion_matrix(y_test,pred))

In [ ]:
# This is a classification exercise on synthetic educational data, not a time-series forecast.